# 05 — Formal validation

Run this after Notebook 00. This notebook evaluates the validation split only.
Use these results to confirm the fixed algorithm before final held-out test reporting.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)
print("PROJECT_ROOT:", PROJECT_ROOT, flush=True)

In [ ]:
from carnivore_reconstruction.timing import TimerLog, status
from carnivore_reconstruction.formal_large_felid import read_table_auto, summarize_with_linear, save_metric_bundle, prepare_formal_task_tables, balanced_sample_task_table
from carnivore_reconstruction.proposed import paper_metrics_for_reporting

MODEL_DIR = PROJECT_ROOT / "outputs" / "pretrained_model"
MODEL_PATH = MODEL_DIR / "pretrained_model.joblib"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "formal_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALIDATION_SPLIT = "validation"
MAX_VALIDATION_TASKS = 800  # balanced cap for manageable formal validation
RANDOM_SEED = 42
KEEP_SCORED = False

timer = TimerLog()

In [ ]:
from carnivore_reconstruction import MotifReconstructionModel, tasks_from_tables

status("Validation 1/4: loading frozen model and formal task table")
with timer.step("load_model_and_task_tables"):
    model = MotifReconstructionModel.load(MODEL_PATH)
    AUTO_N_JOBS = int(os.environ.get('CARNIVORE_N_JOBS', max(1, (os.cpu_count() or 2) - 1)))
    model.config.n_jobs = AUTO_N_JOBS
    model.config.parallel_threshold_tasks = 1
    model.config.evaluate_environmental_exposure = False
    model.config.save_candidate_paths = False
    print(f'Using n_jobs={AUTO_N_JOBS}; environmental exposure/path-row saving disabled for evaluation speed.', flush=True)
    task_table = read_table_auto(MODEL_DIR / "task_table.parquet")
    task_points = read_table_auto(MODEL_DIR / "task_points.parquet")
    split_table_path = MODEL_DIR / "formal_individual_split_table.csv"
    split_table = pd.read_csv(split_table_path) if split_table_path.exists() else None
    task_table, task_points, split_table = prepare_formal_task_tables(
        task_table,
        task_points,
        split_table,
        seed=20260625,
        label="formal_task_table",
    )

print("task_table:", task_table.shape)
display(task_table.groupby(["dataset", "taxon", "setting_name", "split"]).size().reset_index(name="n_tasks"))

In [ ]:
status("Validation 2/4: selecting validation tasks")
selected_table = task_table[task_table["split"].eq(VALIDATION_SPLIT)].copy().reset_index(drop=True)
if selected_table.empty:
    raise ValueError("No validation tasks found. Check the formal split table.")
selected_table = balanced_sample_task_table(selected_table, MAX_VALIDATION_TASKS, seed=RANDOM_SEED, label="formal_validation")

# Existing overnight pretrained models may not yet contain ecological candidate artifacts.
from carnivore_reconstruction.ecological_candidates import ensure_ecological_artifacts
train_table_for_artifacts = balanced_sample_task_table(task_table[task_table["split"].eq("train")].copy(), 3000, seed=RANDOM_SEED, label="artifact_training_support")
train_tasks_for_artifacts = tasks_from_tables(train_table_for_artifacts, task_points)
model = ensure_ecological_artifacts(model, train_tasks_for_artifacts, timer=timer)
selected_tasks = tasks_from_tables(selected_table, task_points)
print(f"Selected {len(selected_tasks):,} validation tasks")

In [ ]:
from carnivore_reconstruction.benchmark import run_accuracy_benchmark

status("Validation 3/4: running publication formal validation benchmark")
with timer.step("run_formal_validation", n_tasks=len(selected_tasks), split=VALIDATION_SPLIT):
    result = run_accuracy_benchmark(model, selected_tasks, keep_scored=KEEP_SCORED, task_table=task_table)

task_metrics = result["task_metrics"]
selected_candidates = result.get("selected_candidates", pd.DataFrame())
runtime = result.get("runtime", pd.DataFrame())
setting_choices = result.get("setting_choices", pd.DataFrame())

print("metrics:", task_metrics.shape)
print("selected_candidates:", selected_candidates.shape)
print("runtime:", runtime.shape)

In [ ]:
status("Validation 4/4: saving formal validation outputs")
with timer.step("save_formal_validation_outputs", n_tasks=len(selected_tasks)):
    save_metric_bundle(task_metrics, OUTPUT_DIR, "formal_validation", paper_filter_func=paper_metrics_for_reporting)
    selected_candidates.to_csv(OUTPUT_DIR / "formal_validation_selected_candidates.csv", index=False)
    runtime.to_csv(OUTPUT_DIR / "formal_validation_runtime.csv", index=False)
    setting_choices.to_csv(OUTPUT_DIR / "formal_validation_setting_choices.csv", index=False)
    timer.save(OUTPUT_DIR / "runtime_formal_validation.csv")

print("Saved outputs to:", OUTPUT_DIR)
display(pd.read_csv(OUTPUT_DIR / "paper_formal_validation_method_summary.csv"))